# 127 — Blackboard y memoria compartida

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Ciclo 1: solo A es aplicable (medio vacío) → s1 (0.8). Ciclo 2: B
aplicable (hay un síntoma), C no (solo 1 síntoma) → s2 (0.7). Ciclo 3: C aplicable →
h1 con conf = (0.8+0.7)/2 = **0.75** ≥ 0.7 → fin en 3 ciclos. Nótese que la prioridad
C > B > A no alteró el orden: las *condiciones* lo forzaron — el control solo
desempata entre aplicables.

**Ejercicio 2.** (a) Última escritura: `h_deploy` fue pisada por `h_pool`; al
refutarse `h_pool` el medio queda sin alternativas y el sistema debe re-derivar desde
cero (si aún puede). (b) Coexistencia: `h_deploy` (0.4) sigue ahí; el control la
promueve como siguiente candidata. La coexistencia compra **recuperación** a cambio de
un medio más grande — de ahí la necesidad de poda con umbral mínimo, no de
sobrescritura.

**Ejercicio 3.** Ver celda: el bucle evalúa condiciones, aplica la de mayor prioridad
y termina por umbral. La traza reproduce el ejercicio 1.

**Ejercicio 4.** Cada worker se mapea a
`{"nivel": "sintoma", "conf": score, "autor": agent, "texto": finding}`. KS `sintesis`:
"condición: existen ≥3 síntomas de autores distintos; acción: escribir en nivel
`decision` la hipótesis 'mejorar <autor del síntoma de conf mínima>' con conf =
1 − conf_mínima". Reproduce la regla del supervisor como una KS más: la jerarquía se
convirtió en coordinación por el medio.


In [ ]:
result = run_lab("multiagent", seed=127)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 3: mini-blackboard con traza
bb = []

def ks_A(bb):
    if not bb:
        return {"id": "s1", "nivel": "sintoma", "conf": 0.8, "autor": "A"}

def ks_B(bb):
    hay_sintoma = any(h["nivel"] == "sintoma" for h in bb)
    ya_escribio = any(h["autor"] == "B" for h in bb)
    if hay_sintoma and not ya_escribio:
        return {"id": "s2", "nivel": "sintoma", "conf": 0.7, "autor": "B"}

def ks_C(bb):
    sintomas = [h for h in bb if h["nivel"] == "sintoma"]
    ya = any(h["nivel"] == "hipotesis" for h in bb)
    if len(sintomas) >= 2 and not ya:
        conf = sum(h["conf"] for h in sintomas) / len(sintomas)
        return {"id": "h1", "nivel": "hipotesis", "conf": round(conf, 3), "autor": "C"}

def control(bb, max_ciclos=6):
    for ciclo in range(1, max_ciclos + 1):
        for ks in (ks_C, ks_B, ks_A):          # prioridad C > B > A
            nueva = ks(bb)
            if nueva:
                bb.append(nueva)
                print(f"ciclo {ciclo}: {ks.__name__} escribe {nueva}")
                break
        else:
            print(f"ciclo {ciclo}: sin KS aplicable → deadlock epistémico")
            return bb
        if any(h["nivel"] == "hipotesis" and h["conf"] >= 0.7 for h in bb):
            print(f"umbral alcanzado en ciclo {ciclo}")
            return bb
    return bb

control(bb)

# Ejercicio 4: laboratorio → estado inicial de blackboard
result = run_lab("multiagent", seed=127)
hipotesis_iniciales = [
    {"nivel": "sintoma", "conf": w["score"], "autor": w["agent"], "texto": w["finding"]}
    for w in result["result"]["workers"]
]
show(hipotesis_iniciales)
peor = min(hipotesis_iniciales, key=lambda h: h["conf"])
print("KS sintesis escribiría: mejorar", peor["autor"])


## Reflexión

1. En el laboratorio los workers entregan sus hallazgos al supervisor y no los ven entre sí. ¿Qué hallazgo concreto de un worker podría cambiar la evaluación de otro si compartieran blackboard (p. ej. security ↔ documentation)?
2. ¿Por qué "las hipótesis incompatibles coexisten con confianza" es mejor política que "gana la última escritura", y qué coste tiene en el tamaño del medio?
3. ¿Qué señal cuantitativa usarías para detectar deadlock epistémico (el sistema itera sin acercarse a la solución) y qué acción dispararías?
